# C1DB1B: Data Management for Decision-Making
## Workshop 4: From Model Results to Defensible Action
### Executive decision: Where should Vårdlyftet establish its next primary care centre?

---

You still advise **Vårdlyftet**. You still compare **five municipalities**. You may still recommend that it should **not** proceed.

Module 3 produced scores and rankings. Module 4 asks what management should **do**.

> Ranking ≠ prediction ≠ feasibility.

No new scoring method is required. Reuse the approved Module 3 functions. Run scenarios on **copies**. Do not overwrite the baseline.

---

### Sequence for this module

1. Trace the evidence and review constraints (Met / Not met / Unknown).
2. Run one operating-assumption scenario (uniform 20% cost increase).
3. Run one combined adverse scenario (cost increase **and** cost-priority weights).
4. State how strong the robustness claim is.
5. Prediction: evaluate a Module 2 hold-out if the group built a model; otherwise mark **Not assigned**.
6. Write Go / Conditional Go / No-Go advice in five sentences.
7. Return to Decision Model Zero and analyse one recorded episode.
8. Prepare the defense.

Complete the **common benchmark rehearsal** first. Then apply the analogous scenarios to **your** Module 3 package: your scored columns, your weights, and your named cost or constraint fields. Do not substitute the demonstration indicator set for the group's architecture.

---

### Data vintage

Paths, the cost multiplier, the teaching ceiling, and the predictive flag come from `data/manifest.json`. Do not hard-code file names.


---
### Who does what

**You, before the workshop:** bring a **laptop**. Rerun the approved Module 3 functions on **your** architecture, complete both issued scenarios on copies, and draft the five-sentence recommendation.

**The teacher, in the lecture:** separates ranking from feasibility and revises examination concepts.

**The teacher, in the workshop:** checks the scenario table, releases a benchmark multiplier or ceiling change, and asks defense questions.

**You, in the workshop:** freeze prepared results, apply the live change, and explain Go / Conditional Go / No-Go.

Recommending a municipality is screening, not a premises contract. The solution notebook is an instructor demonstration for codes 1481, 1490, 1463, 1465, 1443 --- not a recommendation to copy.


---
## Part 1: Load the contract

The benchmark alternatives A--E are **synthetic**. They are not municipalities.

Module 4 issues scenario parameters and a teaching cost ceiling. It does **not** issue a hold-out test set.


In [ ]:
# ==============================================================================
# Load vintage, Module 3 benchmark, and Module 4 scenario contract
# ==============================================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

manifest_path = Path("data/manifest.json")
if not manifest_path.exists():
    raise FileNotFoundError("Run this notebook from the folder that contains data/.")

MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
m1 = MANIFEST["module1"]
m2 = MANIFEST["module2"]
m3 = MANIFEST["module3"]
m4 = MANIFEST["module4"]
T0 = int(m1["t0"])
T1 = int(m1["t1"])
PREDICTIVE_STATUS = m4["predictive_component"]
DOCUMENTED_MISSING = list(m2["documented_missing_markers"])

print(f"Course vintage       : {MANIFEST['vintage']}")
print(f"Module 4 status      : {m4['status']}")
print(f"Predictive component : {PREDICTIVE_STATUS}")
print(f"Scenario file        : {m4['scenario_file']}")

bench = pd.read_csv(m3["benchmark_file"]).set_index("Alternative")
meta = pd.read_csv(m3["benchmark_meta_file"]).set_index("indicator")
meta = meta.reindex(bench.columns)
weights_b = meta["weight"].astype(float)
benefit_b = meta["direction"].eq("Benefit")

spec = pd.read_csv(m4["scenario_file"])
constraints = pd.read_csv(m4["constraint_file"])

print("\nBenchmark matrix:")
print(bench.to_string())
print("\nIssued scenarios:")
print(spec.to_string(index=False))
print("\nIssued constraints:")
print(constraints.to_string(index=False))


---
## Shared scoring functions

Reuse the Module 3 conventions. Do not feed the WSM $Z$ matrix into TOPSIS.


In [ ]:
# ==============================================================================
# Course WSM and TOPSIS conventions (unchanged from Module 3)
# ==============================================================================
def check_ready(X, weights, benefit):
    weights = weights.reindex(X.columns)
    benefit = benefit.reindex(X.columns)
    if not X.index.is_unique:
        raise ValueError("Row identifiers are not unique.")
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError("Scoring matrix contains missing or infinite values.")
    if weights.isna().any() or benefit.isna().any():
        raise ValueError("Every indicator needs a weight and a direction.")
    if (weights < 0).any():
        raise ValueError("Weights must be non-negative.")
    if not np.isclose(float(weights.sum()), 1.0):
        raise ValueError("Weights must sum to 1.")
    return weights, benefit.astype(bool)


def wsm_scores(X, weights, benefit):
    weights, benefit = check_ready(X, weights, benefit)
    span = X.max() - X.min()
    Z = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    constant = []
    for col in X.columns:
        if np.isclose(span[col], 0.0):
            Z[col] = 0.0
            constant.append(col)
        elif benefit[col]:
            Z[col] = (X[col] - X[col].min()) / span[col]
        else:
            Z[col] = (X[col].max() - X[col]) / span[col]
    score = Z.mul(weights, axis="columns").sum(axis=1)
    return score, Z, constant


def topsis_scores(X, weights, benefit):
    weights, benefit = check_ready(X, weights, benefit)
    col_norm = np.sqrt(X.pow(2).sum(axis=0))
    R = X.div(col_norm.where(col_norm.ne(0), 1.0), axis="columns")
    V = R.mul(weights, axis="columns")
    ideal = pd.Series({c: V[c].max() if benefit[c] else V[c].min() for c in X.columns})
    anti = pd.Series({c: V[c].min() if benefit[c] else V[c].max() for c in X.columns})
    d_plus = np.sqrt(V.sub(ideal, axis="columns").pow(2).sum(axis=1))
    d_minus = np.sqrt(V.sub(anti, axis="columns").pow(2).sum(axis=1))
    denom = d_plus + d_minus
    closeness = d_minus.div(denom.where(denom.ne(0)))
    return closeness, R, V, ideal, anti, d_plus, d_minus


def rank_higher_better(score):
    return score.rank(ascending=False, method="average")


def compare_methods(wsm, topsis):
    out = pd.DataFrame({"WSM_score": wsm, "TOPSIS_score": topsis})
    out["WSM_rank"] = rank_higher_better(out["WSM_score"])
    out["TOPSIS_rank"] = rank_higher_better(out["TOPSIS_score"])
    if out[["WSM_rank", "TOPSIS_rank"]].nunique().min() > 1:
        rho = out["WSM_rank"].corr(out["TOPSIS_rank"], method="spearman")
    else:
        rho = np.nan
    return out, rho


def redistribute(weights, indicator, alpha):
    w = weights.astype(float).copy()
    if indicator not in w.index:
        raise KeyError(indicator)
    rest = 1.0 - float(w[indicator])
    if rest <= 0:
        raise ValueError("Cannot redistribute when the selected weight is 1.")
    w[indicator] = float(alpha)
    for k in w.index:
        if k != indicator:
            w[k] = float(weights[k]) * (1.0 - float(alpha)) / rest
    return w


def evaluate_ceiling(cost, ceiling):
    status = pd.Series(np.where(cost <= ceiling, "Met", "Not met"), index=cost.index)
    return pd.DataFrame({"cost": cost, "ceiling": ceiling, "status": status})


def run_models(X, weights, benefit):
    wsm, Z, _ = wsm_scores(X, weights, benefit)
    tops, *_ = topsis_scores(X, weights, benefit)
    cmp, rho = compare_methods(wsm, tops)
    return cmp, rho, Z


---
## Task 1: Trace the evidence and review constraints

Preserve the approved Module 3 matrix. Prefer the five-row table you saved from Module 2 (`data/group_sources/decision_table.csv`) with **your** scored columns. The issued-file reconstruction below is the method demo for the demonstration portfolio. Then record who leads, why, what changes the result, and which constraints are Met / Not met / Unknown.

In this vintage, **do not score** a travel-time field that is missing in source.


In [ ]:
# ==============================================================================
# Reconstruct the demonstration scoring matrix (replace codes with YOUR portfolio)
# ==============================================================================
# TODO: replace with YOUR allocated codes
portfolio_codes = [1481, 1490, 1463, 1465, 1443]
NAME_MAP = {
    "1481": "Molndal",
    "1490": "Boras",
    "1463": "Mark",
    "1465": "Svenljunga",
    "1443": "Bollebygd",
}


def standardize_code(s):
    return s.astype("string").str.strip()


def parse_numeric(series, markers):
    raw = series.astype("string").str.strip()
    cleaned = raw.mask(raw.isin(list(markers)))
    numeric = pd.to_numeric(cleaned, errors="coerce")
    unexpected = cleaned.notna() & numeric.isna()
    if unexpected.any():
        raise ValueError("Unexpected tokens: " + str(cleaned.loc[unexpected].unique()))
    return numeric


anchor = pd.read_csv(m1["anchor_file"])
annual = pd.read_csv(m2["annual_file"], dtype={"Avg_Travel_Time_min": "string"})
static = pd.read_csv(m2["static_file"], dtype={"Est_Premises_Cost_kSEK": "string"})
for frame in (anchor, annual, static):
    frame["Municipality_Code"] = standardize_code(frame["Municipality_Code"])

codes = {str(c) for c in portfolio_codes}
demo = anchor[anchor["Municipality_Code"].isin(codes) & anchor["Year"].eq(T1)].copy()
hist = anchor[anchor["Municipality_Code"].isin(codes) & anchor["Year"].eq(T0)].copy()

annual["Avg_Travel_Time_min_num"] = parse_numeric(annual["Avg_Travel_Time_min"], DOCUMENTED_MISSING)
static["Est_Premises_Cost_kSEK_num"] = parse_numeric(static["Est_Premises_Cost_kSEK"], DOCUMENTED_MISSING)

merged = demo.merge(
    annual[["Municipality_Code", "Year", "Operating_Clinics", "Clinics_per_10k", "Avg_Travel_Time_min_num"]],
    on=["Municipality_Code", "Year"],
    how="left",
    validate="one_to_one",
)
merged = merged.merge(
    static[["Municipality_Code", "Care_Need_Index", "Est_Premises_Cost_kSEK_num"]],
    on="Municipality_Code",
    how="left",
    validate="one_to_one",
)
merged = merged.merge(
    hist[["Municipality_Code", "Pop_80_Plus"]].rename(columns={"Pop_80_Plus": "Pop_80_Plus_t0"}),
    on="Municipality_Code",
    how="left",
    validate="one_to_one",
)

merged["Share_80_plus_pct"] = 100.0 * merged["Pop_80_Plus"] / merged["Pop_Total"]
merged["Dependency_ratio"] = 100.0 * (
    merged["Pop_0_17"] + merged["Pop_65_79"] + merged["Pop_80_Plus"]
) / merged["Pop_18_64"]
merged["Growth_80_plus_pct"] = 100.0 * (
    merged["Pop_80_Plus"] - merged["Pop_80_Plus_t0"]
) / merged["Pop_80_Plus_t0"]

decision = merged.set_index("Municipality_Code")
decision["Municipality_Name"] = decision.index.map(NAME_MAP)

scored = [
    "Share_80_plus_pct",
    "Dependency_ratio",
    "Growth_80_plus_pct",
    "Clinics_per_10k",
    "Care_Need_Index",
    "Est_Premises_Cost_kSEK_num",
]
X_port = decision[scored].astype(float)
benefit_p = pd.Series(
    {
        "Share_80_plus_pct": True,
        "Dependency_ratio": True,
        "Growth_80_plus_pct": True,
        "Clinics_per_10k": False,
        "Care_Need_Index": True,
        "Est_Premises_Cost_kSEK_num": False,
    }
)
weights_p = pd.Series(
    {
        "Share_80_plus_pct": 0.20,
        "Dependency_ratio": 0.15,
        "Growth_80_plus_pct": 0.20,
        "Clinics_per_10k": 0.15,
        "Care_Need_Index": 0.15,
        "Est_Premises_Cost_kSEK_num": 0.15,
    }
)

check_ready(X_port, weights_p, benefit_p)
print("Portfolio scoring matrix (issued companions — method demo):")
print(X_port.round(3).to_string())
print("Travel time excluded from scoring (source missing for at least one alternative).")

# --- Your architecture ---
# Prefer data/group_sources/decision_table.csv from Module 2, then replace
# `scored`, `benefit_p`, and `weights_p` with YOUR approved set before the scenarios.
# own = pd.read_csv(Path("data/group_sources/decision_table.csv"), dtype={"Municipality_Code": "string"}).set_index("Municipality_Code")
# scored = ["Share_80_plus_pct", "YOUR_KPI_2", "YOUR_KPI_3"]
# X_port = own[scored].astype(float)


---
### Your analysis: evidence and constraints

**Who leads under the Module 3 baseline, and where do the methods disagree?**
*(Write here.)*

**Which constraints are Met, Not met, or Unknown? Unknown is not Met.**
*(Write here.)*

**Which values are observed, derived, predicted, or assumed?**
*(Write here.)*



---
## Tasks 2--3: Two stress scenarios on copies

1. **Operating scenario:** multiply the named cost field by 1.20. Keep other indicators and baseline weights unchanged.
2. **Combined scenario:** the same cost scaling **and** the issued cost-priority weights.

Define the change before running it. Recheck the ceiling. Do not overwrite the baseline.

**Self-check:** after a uniform positive cost scaling, ranks may stay the same. Alternative B's benchmark cost should become 7.2 and fail 6.5.


In [ ]:
# ==============================================================================
# TASKS 2-3: Preserve the baseline, then run two stress scenarios on copies
# ==============================================================================
COST_B = m4["benchmark_cost_field"]
MULT_B = float(m4["benchmark_cost_multiplier"])
CEIL_B = float(m4["benchmark_cost_ceiling"])
W_COST_B = float(m4["benchmark_combined_cost_weight"])

COST_P = m4["portfolio_cost_field"]
MULT_P = float(m4["portfolio_cost_multiplier"])
CEIL_P = float(m4["portfolio_cost_ceiling"])
W_COST_P = float(m4["portfolio_combined_cost_weight"])

baseline_Xb = bench.copy(deep=True)
baseline_wb = weights_b.copy()
baseline_Xp = X_port.copy(deep=True)
baseline_wp = weights_p.copy()

cmp_b0, rho_b0, _ = run_models(baseline_Xb, baseline_wb, benefit_b)
cmp_p0, rho_p0, _ = run_models(baseline_Xp, baseline_wp, benefit_p)
con_b0 = evaluate_ceiling(baseline_Xb[COST_B], CEIL_B)
con_p0 = evaluate_ceiling(baseline_Xp[COST_P], CEIL_P)

print("=== S1 baseline (benchmark) ===")
print(cmp_b0.round(4).to_string())
print("Spearman rho:", rho_b0)
print(con_b0.to_string())

print("\n=== S1 baseline (portfolio) ===")
print(cmp_p0.sort_values("WSM_rank").round(4).to_string())
print("Spearman rho:", rho_p0)
print(con_p0.to_string())

# S2: uniform cost increase, weights unchanged
X_b2 = baseline_Xb.copy(deep=True)
X_b2[COST_B] = X_b2[COST_B] * MULT_B
X_p2 = baseline_Xp.copy(deep=True)
X_p2[COST_P] = X_p2[COST_P] * MULT_P

cmp_b2, rho_b2, _ = run_models(X_b2, baseline_wb, benefit_b)
cmp_p2, rho_p2, _ = run_models(X_p2, baseline_wp, benefit_p)
con_b2 = evaluate_ceiling(X_b2[COST_B], CEIL_B)
con_p2 = evaluate_ceiling(X_p2[COST_P], CEIL_P)

print("\n=== S2 operating cost x{:.2f} (benchmark) ===".format(MULT_B))
print("Rank change vs baseline (WSM):", not cmp_b2["WSM_rank"].equals(cmp_b0["WSM_rank"]))
print("Rank change vs baseline (TOPSIS):", not cmp_b2["TOPSIS_rank"].equals(cmp_b0["TOPSIS_rank"]))
print(cmp_b2.round(4).to_string())
print(con_b2.to_string())
print("B cost after shock:", round(float(X_b2.loc["B", COST_B]), 1))

print("\n=== S2 operating cost x{:.2f} (portfolio) ===".format(MULT_P))
print("Rank change vs baseline (WSM):", not cmp_p2["WSM_rank"].equals(cmp_p0["WSM_rank"]))
print("Rank change vs baseline (TOPSIS):", not cmp_p2["TOPSIS_rank"].equals(cmp_p0["TOPSIS_rank"]))
print(cmp_p2.sort_values("WSM_rank").round(4).to_string())
print(con_p2.to_string())

# Confirm the baseline objects were not overwritten
assert baseline_Xb[COST_B].equals(bench[COST_B])
assert baseline_Xp[COST_P].equals(X_port[COST_P])
print("\nBaseline preservation: PASS")

# S3: same cost scaling AND cost-priority weights
w_b3 = baseline_wb.copy()
w_b3["opportunity"] = 1.0 - W_COST_B
w_b3[COST_B] = W_COST_B
assert np.isclose(float(w_b3.sum()), 1.0)

w_p3 = redistribute(baseline_wp, COST_P, W_COST_P)
assert np.isclose(float(w_p3.sum()), 1.0)

cmp_b3, rho_b3, _ = run_models(X_b2.copy(deep=True), w_b3, benefit_b)
cmp_p3, rho_p3, _ = run_models(X_p2.copy(deep=True), w_p3, benefit_p)

print("\n=== S3 combined (benchmark: cost x{:.2f} and cost weight {:.2f}) ===".format(MULT_B, W_COST_B))
print("weights:", {k: round(float(v), 4) for k, v in w_b3.items()})
print(cmp_b3.round(4).to_string())
print("WSM leader:", cmp_b3["WSM_score"].idxmax(), "TOPSIS leader:", cmp_b3["TOPSIS_score"].idxmax())
print("WSM rank change vs S1:", not cmp_b3["WSM_rank"].equals(cmp_b0["WSM_rank"]))
print("TOPSIS rank change vs S1:", not cmp_b3["TOPSIS_rank"].equals(cmp_b0["TOPSIS_rank"]))

print("\n=== S3 combined (portfolio: cost x{:.2f} and premises-cost weight {:.2f}) ===".format(MULT_P, W_COST_P))
print("weights:", {k: round(float(v), 4) for k, v in w_p3.items()})
print(cmp_p3.sort_values("WSM_rank").round(4).to_string())
print("WSM leader:", cmp_p3["WSM_score"].idxmax(), "TOPSIS leader:", cmp_p3["TOPSIS_score"].idxmax())
print("WSM rank change vs S1:", not cmp_p3["WSM_rank"].equals(cmp_p0["WSM_rank"]))
print("TOPSIS rank change vs S1:", not cmp_p3["TOPSIS_rank"].equals(cmp_p0["TOPSIS_rank"]))


---
### Your analysis: operating scenario

**Did the rankings change after the 20% cost increase? Why or why not?**
*(Write here.)*

**Which alternatives cross the teaching ceiling? What does that mean for advice?**
*(Write here.)*



---
### Your analysis: combined scenario

**Which change moved the ranking, and which change moved the constraint status?**
*(Write here.)*

**Do not treat independently normalized scores as fixed-scale business gains or losses.**
*(Write here.)*



---
## Task 4: How strong is the robustness claim?

Show leaders, reversals, and constraint status for S1, S2, and S3. Too strong: "The recommendation is robust."


In [ ]:
# ==============================================================================
# Decision exhibits: ranks across scenarios, and absolute cost vs ceiling
# ==============================================================================
def scenario_rank_table(cmp_s1, cmp_s2, cmp_s3):
    out = pd.DataFrame(
        {
            "WSM_S1": cmp_s1["WSM_rank"],
            "WSM_S2": cmp_s2["WSM_rank"],
            "WSM_S3": cmp_s3["WSM_rank"],
            "TOPSIS_S1": cmp_s1["TOPSIS_rank"],
            "TOPSIS_S2": cmp_s2["TOPSIS_rank"],
            "TOPSIS_S3": cmp_s3["TOPSIS_rank"],
        }
    )
    return out


ranks_b = scenario_rank_table(cmp_b0, cmp_b2, cmp_b3)
ranks_p = scenario_rank_table(cmp_p0, cmp_p2, cmp_p3)
print("Benchmark ranks across scenarios:")
print(ranks_b.to_string())
print("\nPortfolio ranks across scenarios:")
print(ranks_p.to_string())


def plot_ranks(ranks, title):
    fig, ax = plt.subplots(figsize=(8.2, 4.0))
    x = np.arange(len(ranks))
    width = 0.12
    series = [
        ("WSM S1", ranks["WSM_S1"], "#1f4e79"),
        ("WSM S2", ranks["WSM_S2"], "#5b8db8"),
        ("WSM S3", ranks["WSM_S3"], "#9bb7d4"),
        ("TOPSIS S1", ranks["TOPSIS_S1"], "#c0392b"),
        ("TOPSIS S2", ranks["TOPSIS_S2"], "#e67e22"),
        ("TOPSIS S3", ranks["TOPSIS_S3"], "#f5b041"),
    ]
    for i, (label, series_vals, color) in enumerate(series):
        ax.bar(x + (i - 2.5) * width, series_vals, width, label=label, color=color)
    ax.set_xticks(x)
    ax.set_xticklabels(ranks.index.astype(str))
    ax.set_ylabel("Rank (1 = best)")
    ax.set_ylim(0.5, 5.5)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_cost_ceiling(s1, s2, ceiling, title, ylabel):
    fig, ax = plt.subplots(figsize=(7.4, 3.8))
    x = np.arange(len(s1))
    ax.bar(x - 0.18, s1, 0.36, label="Baseline cost", color="#1f4e79")
    ax.bar(x + 0.18, s2, 0.36, label="After 20% increase", color="#c0392b")
    ax.axhline(ceiling, color="black", linestyle="--", linewidth=1.2, label=f"Ceiling {ceiling:g}")
    ax.set_xticks(x)
    ax.set_xticklabels(s1.index.astype(str))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()


plot_ranks(ranks_b, "Benchmark ranks: baseline, operating shock, combined")
plot_cost_ceiling(
    baseline_Xb[COST_B],
    X_b2[COST_B],
    CEIL_B,
    "Benchmark annual cost versus teaching ceiling 6.5",
    "Annual cost (units)",
)
plot_ranks(ranks_p, "Portfolio ranks: baseline, operating shock, combined")
plot_cost_ceiling(
    baseline_Xp[COST_P],
    X_p2[COST_P],
    CEIL_P,
    "Portfolio premises cost versus illustrative ceiling 650 kSEK",
    "Est. premises cost (kSEK)",
)


---
### Your analysis: robustness

**Write a defensible robustness claim. Name what was tested and what was not.**
*(Write here.)*



---
## Task 5: Predictive evaluation, if the group took it up

Keep final testing separate from stress scenarios. If the group built a model in Module 2, this is where the held-out 20% may be evaluated. If the group used an external forecast, keep it labelled. If not taken up, mark **Not assigned**.


In [ ]:
# ==============================================================================
# TASK 5: Predictive component status
# ==============================================================================
print("Issued predictive flag (common path):", PREDICTIVE_STATUS)
print("If the group built a model: evaluate the reserved 20% here; do not retune.")
print("If the group used an external forecast: keep it labelled; do not invent a hold-out.")
print("If the group did not take up prediction: mark Not assigned.")
print("Stress scenarios are not a substitute for a predictive test.")


---
## Task 6: From evidence to action

Choose **Go**, **Conditional Go**, or **No-Go**. Go does not mean open the clinic tomorrow.

Write five sentences a manager can use without opening this notebook. Conditions must name evidence, a decision point, and a consequence.


---
### Your analysis: recommendation

**Advice (Go / Conditional Go / No-Go):**
*(Write here.)*

**Five sentences a manager can understand without opening the notebook:**
1.
2.
3.
4.
5.

**One condition that specifies evidence, a decision point, and a consequence if it fails:**
*(Write here.)*



---
## Task 7: Process reflection (Part B)

Analyse recorded episodes, not personalities. Return to the unchanged Decision Model Zero note. Compare processes, not unlike rankings.


---
### Your analysis: process reflection

Use your group's records. Do not invent an episode.

**Decision Model Zero compared with the municipality recommendation (processes, not unlike rankings):**
*(Write here.)*

**One selected episode:** what was decided, what evidence was used, what disagreement occurred, how it was closed, and which assigned reading helps interpret it.
*(Write here.)*



---
## Task 8: Prepare the defense

Lead with the advice. Be ready for: why this municipality rather than the runner-up; what would reverse the recommendation; whether costs are observed, estimated, or assumed; which constraint is still Unknown.


---
### Your analysis: defense

**Strongest competitor and the assumption that would reverse the advice:**
*(Write here.)*

**One answer using: Answer → Evidence → Boundary → Action**
*(Write here.)*



---
## Live understanding rehearsal (benchmark copy)

The instructor may change the multiplier, the ceiling, or both. Preserve prepared results. This cell rehearses a ceiling change from 6.5 to 7.5 after the 20% cost shock.


In [ ]:
# ==============================================================================
# Live-challenge rehearsal: change the BENCHMARK COPY only
# ==============================================================================
# The instructor may change the multiplier, the ceiling, or both.
# Rehearsal here: keep the 20% shock and raise the ceiling from 6.5 to 7.5.

LIVE_CEILING = 7.5  # TODO: replace with the released value in class
live_X = baseline_Xb.copy(deep=True)
live_X[COST_B] = live_X[COST_B] * MULT_B
live_cmp, live_rho, _ = run_models(live_X, baseline_wb, benefit_b)
live_con = evaluate_ceiling(live_X[COST_B], LIVE_CEILING)

print("Prediction before running:")
print("Ranks should stay unchanged under uniform positive scaling.")
print("Constraint status CAN change when only the ceiling changes.")
print("\nRehearsal ceiling:", LIVE_CEILING)
print(live_cmp.round(4).to_string())
print(live_con.to_string())
print("WSM ranks unchanged vs S2:", live_cmp["WSM_rank"].equals(cmp_b2["WSM_rank"]))
print("B status at 6.5:", con_b2.loc["B", "status"], "| B status at 7.5:", live_con.loc["B", "status"])
print("Baseline still preserved:", baseline_Xb[COST_B].equals(bench[COST_B]))


---
### Your analysis: live challenge

**What did you predict before the live change, and what actually happened?**
*(Write here.)*



---
## Gateway 4 closing check

- Baseline preserved; scenarios run on copies.
- Ranking, prediction, and feasibility claims are not mixed.
- Advice is Go / Conditional Go / No-Go, with a verifiable condition.
- Prediction is **Not assigned**, or a Module 2 hold-out / labelled forecast is evaluated here.


---
### Your analysis: closing check

**Can a manager understand your five sentences without the notebook?**
*(Write here.)*

